In [1]:
# Baseline 00 v2: prevalence baselines under the final six-fold expanding-window CV protocol
# Saves outputs as CSV files for easier viewing in Jupyter

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

SPLIT_SUMMARY_CSV = BASE_DIR / "Baseline_00_v2_Split_Summary.csv"
FF17_RATES_CSV = BASE_DIR / "Baseline_00_v2_FF17_Group_Rates_FullDev.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Baseline_00_v2_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Baseline_00_v2_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Baseline_00_v2_Predictions.csv"

TARGET_COL = "targeted_in_year"

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. FINAL v2 SPLIT PROTOCOL
# =========================================================
CV_FOLDS = [
    {"fold_id": "Fold_1", "train_start": 2010, "train_end": 2015, "val_year": 2016},
    {"fold_id": "Fold_2", "train_start": 2010, "train_end": 2016, "val_year": 2017},
    {"fold_id": "Fold_3", "train_start": 2010, "train_end": 2017, "val_year": 2018},
    {"fold_id": "Fold_4", "train_start": 2010, "train_end": 2018, "val_year": 2019},
    {"fold_id": "Fold_5", "train_start": 2010, "train_end": 2019, "val_year": 2020},
    {"fold_id": "Fold_6", "train_start": 2010, "train_end": 2020, "val_year": 2021},
]

FINAL_DEV_START = 2010
FINAL_DEV_END = 2021
FINAL_TEST_START = 2022
FINAL_TEST_END = 2024

EXPECTED_SPLIT_COUNTS = {
    "Fold_1": {"train_rows": 24053, "train_pos": 906, "val_rows": 3888, "val_pos": 194},
    "Fold_2": {"train_rows": 27941, "train_pos": 1100, "val_rows": 3811, "val_pos": 208},
    "Fold_3": {"train_rows": 31752, "train_pos": 1308, "val_rows": 3794, "val_pos": 212},
    "Fold_4": {"train_rows": 35546, "train_pos": 1520, "val_rows": 3805, "val_pos": 168},
    "Fold_5": {"train_rows": 39351, "train_pos": 1688, "val_rows": 3907, "val_pos": 209},
    "Fold_6": {"train_rows": 43258, "train_pos": 1897, "val_rows": 4535, "val_pos": 229},
}
EXPECTED_FINAL_TEST = {"test_rows": 13147, "test_pos": 861}

# Sanity check unique firm-years
dupes = df.duplicated(subset=["permno", "year"]).sum()
if dupes != 0:
    raise ValueError(f"Found {dupes} duplicate permno-year rows in the frozen panel.")

# =========================================================
# 4. SPLIT VALIDATION
# =========================================================
split_rows = []

for fold in CV_FOLDS:
    train_mask = (df["year"] >= fold["train_start"]) & (df["year"] <= fold["train_end"])
    val_mask = df["year"] == fold["val_year"]

    train_rows = int(train_mask.sum())
    train_pos = int(df.loc[train_mask, TARGET_COL].sum())
    val_rows = int(val_mask.sum())
    val_pos = int(df.loc[val_mask, TARGET_COL].sum())

    expected = EXPECTED_SPLIT_COUNTS[fold["fold_id"]]

    split_rows.append(
        {
            "fold_id": fold["fold_id"],
            "train_years": f"{fold['train_start']}-{fold['train_end']}",
            "validate_year": fold["val_year"],
            "train_rows": train_rows,
            "train_positives": train_pos,
            "train_positive_rate": float(df.loc[train_mask, TARGET_COL].mean()),
            "validate_rows": val_rows,
            "validate_positives": val_pos,
            "validate_positive_rate": float(df.loc[val_mask, TARGET_COL].mean()),
            "expected_train_rows": expected["train_rows"],
            "expected_train_pos": expected["train_pos"],
            "expected_validate_rows": expected["val_rows"],
            "expected_validate_pos": expected["val_pos"],
            "train_rows_match_expected": int(train_rows == expected["train_rows"]),
            "train_pos_match_expected": int(train_pos == expected["train_pos"]),
            "validate_rows_match_expected": int(val_rows == expected["val_rows"]),
            "validate_pos_match_expected": int(val_pos == expected["val_pos"]),
        }
    )

test_mask = (df["year"] >= FINAL_TEST_START) & (df["year"] <= FINAL_TEST_END)
test_rows = int(test_mask.sum())
test_pos = int(df.loc[test_mask, TARGET_COL].sum())

split_rows.append(
    {
        "fold_id": "Final_Test",
        "train_years": f"{FINAL_DEV_START}-{FINAL_DEV_END}",
        "validate_year": f"{FINAL_TEST_START}-{FINAL_TEST_END}",
        "train_rows": int(((df["year"] >= FINAL_DEV_START) & (df["year"] <= FINAL_DEV_END)).sum()),
        "train_positives": int(df.loc[(df["year"] >= FINAL_DEV_START) & (df["year"] <= FINAL_DEV_END), TARGET_COL].sum()),
        "train_positive_rate": float(df.loc[(df["year"] >= FINAL_DEV_START) & (df["year"] <= FINAL_DEV_END), TARGET_COL].mean()),
        "validate_rows": test_rows,
        "validate_positives": test_pos,
        "validate_positive_rate": float(df.loc[test_mask, TARGET_COL].mean()),
        "expected_train_rows": np.nan,
        "expected_train_pos": np.nan,
        "expected_validate_rows": EXPECTED_FINAL_TEST["test_rows"],
        "expected_validate_pos": EXPECTED_FINAL_TEST["test_pos"],
        "train_rows_match_expected": np.nan,
        "train_pos_match_expected": np.nan,
        "validate_rows_match_expected": int(test_rows == EXPECTED_FINAL_TEST["test_rows"]),
        "validate_pos_match_expected": int(test_pos == EXPECTED_FINAL_TEST["test_pos"]),
    }
)

split_summary_df = pd.DataFrame(split_rows)
split_summary_df.to_csv(SPLIT_SUMMARY_CSV, index=False)

# =========================================================
# 5. FF17 HANDLING FOR BASELINES
# =========================================================
FF17_CODE_TO_SHORT = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}

def ff17_bucket_from_code(series: pd.Series) -> pd.Series:
    mapped = pd.to_numeric(series, errors="coerce").round().astype("Int64").map(FF17_CODE_TO_SHORT)
    return mapped.fillna("Unclassified").astype(str)

df["ff17_group_for_baseline"] = ff17_bucket_from_code(df["ff17_code"])

# =========================================================
# 6. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true, y_prob) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 7. SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_rows = []
prediction_frames = []

for fold in CV_FOLDS:
    train_df = df[(df["year"] >= fold["train_start"]) & (df["year"] <= fold["train_end"])].copy()
    val_df = df[df["year"] == fold["val_year"]].copy()

    y_train = train_df[TARGET_COL].to_numpy()
    y_val = val_df[TARGET_COL].to_numpy()

    global_prevalence = float(y_train.mean())

    # B1: global prevalence
    p_global = np.full(len(val_df), global_prevalence, dtype=float)
    metrics_global = evaluate_predictions(y_val, p_global)
    cv_rows.append(
        {
            "model_id": "B1",
            "model_name": "Global_Prevalence",
            "fold_id": fold["fold_id"],
            "train_years": f"{fold['train_start']}-{fold['train_end']}",
            "validate_year": fold["val_year"],
            **metrics_global,
        }
    )

    prediction_frames.append(
        pd.DataFrame(
            {
                "model_id": "B1",
                "model_name": "Global_Prevalence",
                "evaluation_stage": "dev_cv",
                "fold_id": fold["fold_id"],
                "train_years": f"{fold['train_start']}-{fold['train_end']}",
                "evaluation_year": val_df["year"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "y_true": y_val,
                "predicted_probability": p_global,
            }
        )
    )

    # B2: FF17 grouped prevalence
    group_rates = (
        train_df.groupby("ff17_group_for_baseline", dropna=False)[TARGET_COL]
        .agg(rows="size", positives="sum", positive_rate="mean")
        .reset_index()
    )
    group_rate_map = group_rates.set_index("ff17_group_for_baseline")["positive_rate"]
    p_ff17 = val_df["ff17_group_for_baseline"].map(group_rate_map).fillna(global_prevalence).to_numpy()
    metrics_ff17 = evaluate_predictions(y_val, p_ff17)

    cv_rows.append(
        {
            "model_id": "B2",
            "model_name": "FF17_Group_Prevalence",
            "fold_id": fold["fold_id"],
            "train_years": f"{fold['train_start']}-{fold['train_end']}",
            "validate_year": fold["val_year"],
            **metrics_ff17,
        }
    )

    prediction_frames.append(
        pd.DataFrame(
            {
                "model_id": "B2",
                "model_name": "FF17_Group_Prevalence",
                "evaluation_stage": "dev_cv",
                "fold_id": fold["fold_id"],
                "train_years": f"{fold['train_start']}-{fold['train_end']}",
                "evaluation_year": val_df["year"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "ff17_group_for_baseline": val_df["ff17_group_for_baseline"].to_numpy(),
                "y_true": y_val,
                "predicted_probability": p_ff17,
            }
        )
    )

cv_metrics_df = pd.DataFrame(cv_rows)
cv_metrics_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 8. FINAL UNTOUCHED TEST (FIT ON 2010-2021 ONLY)
# =========================================================
dev_df = df[(df["year"] >= FINAL_DEV_START) & (df["year"] <= FINAL_DEV_END)].copy()
test_df = df[(df["year"] >= FINAL_TEST_START) & (df["year"] <= FINAL_TEST_END)].copy()

y_dev = dev_df[TARGET_COL].to_numpy()
y_test = test_df[TARGET_COL].to_numpy()

global_dev_prevalence = float(y_dev.mean())

full_dev_group_rates = (
    dev_df.groupby("ff17_group_for_baseline", dropna=False)[TARGET_COL]
    .agg(rows="size", positives="sum", positive_rate="mean")
    .reset_index()
    .sort_values(["positive_rate", "rows"], ascending=[False, False])
    .reset_index(drop=True)
)
full_dev_group_rates.to_csv(FF17_RATES_CSV, index=False)

full_dev_group_rate_map = full_dev_group_rates.set_index("ff17_group_for_baseline")["positive_rate"]

# B1 final test
p_global_test = np.full(len(test_df), global_dev_prevalence, dtype=float)
metrics_global_test = evaluate_predictions(y_test, p_global_test)

# B2 final test
p_ff17_test = test_df["ff17_group_for_baseline"].map(full_dev_group_rate_map).fillna(global_dev_prevalence).to_numpy()
metrics_ff17_test = evaluate_predictions(y_test, p_ff17_test)

prediction_frames.append(
    pd.DataFrame(
        {
            "model_id": "B1",
            "model_name": "Global_Prevalence",
            "evaluation_stage": "final_test",
            "fold_id": "Final_Test",
            "train_years": f"{FINAL_DEV_START}-{FINAL_DEV_END}",
            "evaluation_year": test_df["year"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "y_true": y_test,
            "predicted_probability": p_global_test,
        }
    )
)

prediction_frames.append(
    pd.DataFrame(
        {
            "model_id": "B2",
            "model_name": "FF17_Group_Prevalence",
            "evaluation_stage": "final_test",
            "fold_id": "Final_Test",
            "train_years": f"{FINAL_DEV_START}-{FINAL_DEV_END}",
            "evaluation_year": test_df["year"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "ff17_group_for_baseline": test_df["ff17_group_for_baseline"].to_numpy(),
            "y_true": y_test,
            "predicted_probability": p_ff17_test,
        }
    )
)

def summarize_model(cv_df: pd.DataFrame, model_id: str, model_name: str, final_test_metrics: dict) -> dict:
    return {
        "model_id": model_id,
        "model_name": model_name,
        "cv_mean_pr_auc": float(cv_df["pr_auc"].mean()),
        "cv_std_pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
        "cv_mean_roc_auc": float(cv_df["roc_auc"].mean()),
        "cv_std_roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
        "cv_mean_brier_score": float(cv_df["brier_score"].mean()),
        "cv_std_brier_score": float(cv_df["brier_score"].std(ddof=1)),
        "final_test_pr_auc": float(final_test_metrics["pr_auc"]),
        "final_test_roc_auc": float(final_test_metrics["roc_auc"]),
        "final_test_brier_score": float(final_test_metrics["brier_score"]),
    }

stage_metrics_df = pd.DataFrame(
    [
        summarize_model(
            cv_metrics_df[cv_metrics_df["model_id"] == "B1"].copy(),
            "B1",
            "Global_Prevalence",
            metrics_global_test,
        ),
        summarize_model(
            cv_metrics_df[cv_metrics_df["model_id"] == "B2"].copy(),
            "B2",
            "FF17_Group_Prevalence",
            metrics_ff17_test,
        ),
    ]
)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 9. DISPLAY RESULTS
# =========================================================
print("Saved:")
print(SPLIT_SUMMARY_CSV)
print(FF17_RATES_CSV)
print(CV_FOLD_METRICS_CSV)
print(STAGE_METRICS_CSV)
print(PREDICTIONS_CSV)

print("\nSplit summary:")
print(split_summary_df.to_string(index=False))

print("\nStage metrics:")
print(stage_metrics_df.to_string(index=False))

print("\nDevelopment CV fold metrics:")
print(cv_metrics_df.to_string(index=False))

Saved:
Baseline_00_v2_Split_Summary.csv
Baseline_00_v2_FF17_Group_Rates_FullDev.csv
Baseline_00_v2_DevCV_Fold_Metrics.csv
Baseline_00_v2_Stage_Metrics.csv
Baseline_00_v2_Predictions.csv

Split summary:
   fold_id train_years validate_year  train_rows  train_positives  train_positive_rate  validate_rows  validate_positives  validate_positive_rate  expected_train_rows  expected_train_pos  expected_validate_rows  expected_validate_pos  train_rows_match_expected  train_pos_match_expected  validate_rows_match_expected  validate_pos_match_expected
    Fold_1   2010-2015          2016       24053              906             0.037667           3888                 194                0.049897              24053.0               906.0                    3888                    194                        1.0                       1.0                             1                            1
    Fold_2   2010-2016          2017       27941             1100             0.039369           3811     